# Build a past-only articulation state

The model predicts knee, ankle, heel and foot-index coordinates relative to the pelvis and prefix torso scale. Both hips already contribute to the shared reference. Own-leg means own distal articulation conditional on that reference.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/restart_data.py').exists()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/restart_data.py').exists())
sys.path.insert(0, str(PROJECT))
from worldmodels.settings import configure
PROJECT, REPOSITORY = configure()
DATA = PROJECT/'artifacts/runs/restart-full10hz-v1'
RUN = PROJECT/'artifacts/runs/restart-dynamics-v1'
assert (DATA/'preparation.json').exists(), 'Run the full extraction command in README first.'
meta = pd.read_csv(DATA/'windows.csv')
with np.load(DATA/'windows.npz', allow_pickle=False) as z:
    x, valid = z['x'], z['valid']
print('Real GAVD observations:', x.shape, 'sources:', meta.source_id.nunique())


Observe the actual valid-joint mask. Context and future each require adequate support.

In [ ]:
fig,ax=plt.subplots(figsize=(10,3));ax.imshow(valid[0].T,aspect='auto',cmap='Greens',vmin=0,vmax=1)
ax.axvline(15.5,color='orange');ax.set(xlabel='Sample index',ylabel='Declared joint index',title='Actual observed support; orange line starts the forecast');plt.show()

Features must be invariant to every future coordinate and future validity bit.

In [ ]:
from worldmodels.restart_experiment import features
f=features(x[:3],valid[:3],0,'bilateral')[0]
changed=x[:3].copy();changed[:,16:]=12345
changed_valid=valid[:3].copy();changed_valid[:,16:]=~changed_valid[:,16:]
np.testing.assert_array_equal(f,features(changed,changed_valid,0,'bilateral')[0])
print('Future-coordinate and future-mask invariance passed:',f.shape)

A temporal control must alter alignment while preserving the available values. A single global permutation can be learned by a linear model and is insufficient.

In [ ]:
print((PROJECT/'docs/restart/protocol.md').read_text().split('## Control-design correction')[1].split('## Common')[0])
print('Available observed context:',16,'frames; forecast:',8,'frames')

A zero is only a storage convention when the validity mask is false. No future target is imputed into observed evidence.

In [ ]:
from worldmodels.restart_experiment import targets
residual,mask=targets(x[:5],valid[:5],0)
print('Target residual shape:',residual.shape,'observed coordinates:',mask.sum())
assert residual.shape==mask.shape